# NjengaData — 03 Charts
**Stacey Koigu | Visualisation**

Mohamed's notebook found the numbers. This one makes them visible.

Four charts, one per finding. My rule for each one: a judge who has never seen this data
should be able to read it in ten seconds without me explaining it.

All charts pull directly from `njenga.db` and save to `data/processed/charts/`.

## Setup

Imports, database connection, and a shared colour palette.
I define the palette once here so all four charts stay consistent —
same blues, same highlight colour, same font.

In [ ]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Move to project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

conn = sqlite3.connect('data/njenga.db')

def q(sql):
    return pd.read_sql(sql, conn)

# Consistent style across all four charts
sns.set_theme(
    style='ticks',
    font='sans-serif',
    rc={
        'axes.spines.top':   False,
        'axes.spines.right': False,
        'figure.dpi':        150,
        'font.size':         11,
    }
)

# Colour palette — same colours used throughout
C = {
    'construction':    '#D85A30',
    'finance':         '#C0392B',
    'professional':    '#8E44AD',
    'preconstruction': '#2E86C1',
    'overheads':       '#95A5A6',
    'nairobi':         '#2E86C1',
    'coast':           '#D85A30',
    'western':         '#27AE60',
    'cement':          '#2E86C1',
    'steel':           '#D85A30',
    'hardcore':        '#27AE60',
    'highlight':       '#D85A30',
    'neutral':         '#2E86C1',
}

# Output folder for chart images
CHARTS = Path('data/processed/charts')
CHARTS.mkdir(parents=True, exist_ok=True)

print('Ready. Charts will save to:', CHARTS.resolve())

## Chart 1 — Where does the money go?

A stacked bar showing how a 2BR build cost splits across categories.
The thing I want a judge to see immediately: the grey pre-construction block
— land, infrastructure, compliance — is bigger than most people expect.
That money is spent before a single wall goes up.

Data: CAHF 2022.

In [ ]:
df1 = q('''
    SELECT cost_category, amount_kes, pct_of_total
    FROM cahf_cost_breakdown
    WHERE unit_type = '2BR_lowrise'
    ORDER BY pct_of_total DESC
''')

# Group the detailed categories into five buckets for the chart
group_map = {
    'Construction':            'Construction',
    'Finance':                 'Finance',
    'Professional fees':       'Professional fees',
    'Land':                    'Pre-construction',
    'Infrastructure':          'Pre-construction',
    'Compliance':              'Pre-construction',
    'Developer overhead':      'Overheads',
    'Marketing':               'Overheads',
    'Other development costs': 'Overheads',
}
bucket_colours = {
    'Construction':     C['construction'],
    'Finance':          C['finance'],
    'Professional fees':C['professional'],
    'Pre-construction': C['preconstruction'],
    'Overheads':        C['overheads'],
}

df1['bucket'] = df1['cost_category'].map(group_map)
df1g = df1.groupby('bucket', as_index=False).agg(
    amount_kes   = ('amount_kes',   'sum'),
    pct_of_total = ('pct_of_total', 'sum')
).sort_values('amount_kes', ascending=False)

# Build the stacked horizontal bar
fig, ax = plt.subplots(figsize=(13, 4))
left = 0
handles = []

for _, row in df1g.iterrows():
    colour = bucket_colours[row['bucket']]
    ax.barh(0, row['amount_kes'], left=left, height=0.5,
            color=colour, edgecolor='white', linewidth=2)
    pct = row['pct_of_total']
    # Only label segments that are wide enough to read
    if pct >= 12:
        ax.text(left + row['amount_kes'] / 2, 0,
                f"{row['bucket']}\n{pct:.1f}%",
                ha='center', va='center',
                fontsize=10, color='white', fontweight='bold')
    elif pct >= 7:
        ax.text(left + row['amount_kes'] / 2, 0,
                f"{pct:.1f}%",
                ha='center', va='center',
                fontsize=9, color='white', fontweight='bold')
    left += row['amount_kes']
    handles.append(mpatches.Patch(color=colour, label=f"{row['bucket']}  ({pct:.1f}%)"))

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'KES {v/1e6:.1f}M'))
ax.set_xlim(0, df1g['amount_kes'].sum() * 1.01)
ax.set_yticks([])
ax.set_xlabel('Total Development Cost (KES)', fontsize=10)
ax.set_title(
    'Where Does the Money Go?\nNairobi 2-Bedroom Low-rise Apartment — Total Cost Breakdown',
    fontsize=13, fontweight='bold', pad=12)
ax.legend(handles=handles, loc='upper center',
          bbox_to_anchor=(0.5, -0.22), ncol=5, frameon=False, fontsize=9)
ax.text(0.01, -0.38, 'Source: CAHF Housing Development Cost Benchmark Kenya 2022',
        transform=ax.transAxes, fontsize=8, color='#888888')

plt.tight_layout()
fig.subplots_adjust(bottom=0.30)
plt.savefig(CHARTS / 'chart_1_cost_breakdown.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_1_cost_breakdown.png')

## Chart 2 — Are costs rising?

Three materials, tracked quarterly since 2019, plotted against a base of 100.
The further a line climbs above 100, the more expensive that material has become.
Steel is the one that moves most — I annotate it directly so the number is impossible to miss.

Data: KNBS Construction Input Price Index.

In [ ]:
df2 = q('''
    SELECT product, year, quarter, index_value
    FROM knbs_material_index
    WHERE product IN ('Cement', 'Steel and reinforced bars', 'Hardcore')
    ORDER BY product, year, quarter
''')

# Annual averages are smoother and easier to read than quarterly dots
df2a = df2.groupby(['product', 'year'])['index_value'].mean().reset_index()
df2a.columns = ['product', 'year', 'avg_index']

# Calculate how much each material has changed from the base of 100
pct = {}
for p, g in df2a.groupby('product'):
    pct[p] = round(g.sort_values('year')['avg_index'].iloc[-1] - 100, 1)

line_colours = {
    'Cement':                    C['cement'],
    'Steel and reinforced bars': C['steel'],
    'Hardcore':                  C['hardcore'],
}
short_names = {
    'Cement':                    'Cement',
    'Steel and reinforced bars': 'Steel',
    'Hardcore':                  'Hardcore',
}

fig, ax = plt.subplots(figsize=(12, 5))

# Shade the supply chain disruption period
ax.axvspan(2021.7, 2022.3, color='#F39C12', alpha=0.12, zorder=0)
ax.text(2022, 185, 'Post-COVID\nsupply chain spike',
        ha='center', fontsize=8, color='#E67E22', style='italic')

for product, group in df2a.groupby('product'):
    group  = group.sort_values('year')
    colour = line_colours[product]
    name   = short_names[product]
    ax.plot(group['year'], group['avg_index'],
            marker='o', markersize=6, linewidth=2.5, color=colour,
            label=f"{name}  (+{pct[product]:.0f}% since 2019)")
    # Label the final value at the end of each line
    ax.annotate(
        f"{group['avg_index'].iloc[-1]:.0f}",
        xy=(group['year'].iloc[-1], group['avg_index'].iloc[-1]),
        xytext=(7, 0), textcoords='offset points',
        fontsize=9, color=colour, fontweight='bold')

ax.axhline(100, color='#CCCCCC', linewidth=1, linestyle='--', label='Base: Dec 2019 = 100')
ax.set_xlabel('Year', fontsize=10)
ax.set_ylabel('Price Index (Dec 2019 = 100)', fontsize=10)
ax.set_xticks(sorted(df2a['year'].unique()))
ax.set_ylim(85, 200)
ax.set_title(
    'Are Construction Costs Rising?\nKey material price indices — annual averages',
    fontsize=13, fontweight='bold', pad=12)
ax.legend(loc='upper left', frameon=False, fontsize=9)
ax.text(0.01, -0.10, 'Source: KNBS Construction Input Price Index 2021–2025',
        transform=ax.transAxes, fontsize=8, color='#888888')

plt.tight_layout()
plt.savefig(CHARTS / 'chart_2_price_trends.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_2_price_trends.png')

## Chart 3 — Is it cheaper to build elsewhere?

Grouped bars comparing cost per m2 across three regions over four years.
This is the counterintuitive chart — people expect the Coast to be cheaper.
The bars say otherwise. I let that land without commentary on the chart itself.

Data: Integrum 2021-2024.

In [ ]:
df3 = q('''
    SELECT year, region, cost_per_m2
    FROM integrum_regional_costs
    WHERE cost_per_m2 IS NOT NULL AND year <= 2024
    ORDER BY region, year
''')

region_colours = {
    'Nairobi/Mt Kenya': C['nairobi'],
    'Coast':            C['coast'],
    'Western/Nyanza':   C['western'],
}
region_labels = {
    'Nairobi/Mt Kenya': 'Nairobi /\nMt Kenya',
    'Coast':            'Coast',
    'Western/Nyanza':   'Western /\nNyanza',
}

regions = df3['region'].unique()
years   = sorted(df3['year'].unique())
n       = len(years)
width   = 0.18
x       = range(len(regions))

fig, ax = plt.subplots(figsize=(11, 5))

for j, year in enumerate(years):
    offsets = [xi + (j - n/2 + 0.5) * width for xi in x]
    vals = []
    for region in regions:
        row = df3[(df3['region']==region) & (df3['year']==year)]
        vals.append(row['cost_per_m2'].values[0] if len(row) else 0)

    # Later years appear lighter — shows direction of change
    alpha = 0.55 + (j / (n - 1)) * 0.45
    colours = [region_colours[r] for r in regions]
    bars = ax.bar(offsets, vals, width=width, color=colours, alpha=alpha,
                  edgecolor='white', linewidth=0.8, label=str(year))
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 400,
                    f'{int(val):,}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(list(x))
ax.set_xticklabels([region_labels[r] for r in regions], fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'KES {int(v):,}'))
ax.set_ylabel('Construction cost per m² (KES)', fontsize=10)
ax.set_title(
    'Is It Cheaper to Build Outside Nairobi?\nStandard bungalow cost per m² by region — 2021 to 2024',
    fontsize=13, fontweight='bold', pad=12)
ax.legend(title='Year', loc='upper left', frameon=False, fontsize=9)
ax.text(0.01, -0.10, 'Source: Integrum Construction annual cost reports 2021–2024',
        transform=ax.transAxes, fontsize=8, color='#888888')

plt.tight_layout()
plt.savefig(CHARTS / 'chart_3_regional_costs.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_3_regional_costs.png')

## Chart 4 — Can James afford to build?

Horizontal bars, one county per row, showing years of full household income needed to build.
Nairobi is highlighted so the judge knows immediately which bar is James's situation.
I add a dotted reference line at 5 years to give a sense of scale — most people feel
that five years of income is already a long time. Several counties are above it.

Data: CAHF 2022 (build cost) + KCHS 2022 (income by county).

In [ ]:
df4 = q('''
    SELECT county, hh_monthly_kes,
           ROUND(3015486.0 / hh_annual_kes, 1) AS years_to_build
    FROM county_income_urban
    WHERE county IN ('Nairobi','Mombasa','Nakuru','Kiambu','Kisumu')
    AND is_national = 0
    ORDER BY years_to_build DESC
''')

# Highlight Nairobi in a different colour
bar_colours = [C['highlight'] if c == 'Nairobi' else C['neutral'] for c in df4['county']]

fig, ax = plt.subplots(figsize=(11, 4.5))

bars = ax.barh(df4['county'], df4['years_to_build'],
               color=bar_colours, edgecolor='white', linewidth=0.8)

for bar, (_, row) in zip(bars, df4.iterrows()):
    ax.text(
        bar.get_width() + 0.08,
        bar.get_y() + bar.get_height() / 2,
        f"{row['years_to_build']} yrs  |  HH income KES {int(row['hh_monthly_kes']):,}/mo",
        va='center', fontsize=9)

ax.set_xlabel('Years of total household income needed', fontsize=10)
ax.set_xlim(0, df4['years_to_build'].max() + 3)
ax.axvline(5, color='#CCCCCC', linewidth=1.2, linestyle='--')
ax.text(5.08, -0.6, '5 years', fontsize=8, color='#AAAAAA')
ax.set_title(
    'Can James Afford to Build Now?\nYears of household income needed — 2BR low-rise apartment (KES 3,015,486)',
    fontsize=13, fontweight='bold', pad=12)
ax.text(-0.01, -0.18,
        'Sources: CAHF HDCB Kenya 2022  |  KCHS Kenya 2022  '
        '(median expenditure × 3.9 household size × 12 months)',
        transform=ax.transAxes, fontsize=8, color='#888888', ha='left')

plt.tight_layout()
plt.savefig(CHARTS / 'chart_4_affordability.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_4_affordability.png')

## Done

All four charts saved as PNGs in `data/processed/charts/`.
The cell below confirms file sizes — if a chart is 0 KB something went wrong upstream.

In [ ]:
charts = sorted(CHARTS.glob('*.png'))
print(f'All charts saved ({len(charts)}):')
for c in charts:
    print(f'  {c.name:45}  {c.stat().st_size // 1024} KB')

conn.close()
print('\nDashboard complete.')